In [68]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import root_mean_squared_error

DATA_DIR = "./data"
for dirname, _, filenames in os.walk(DATA_DIR):
    for filename in filenames:
        print(os.path.join(dirname, filename))

./data/train.csv
./data/test.csv
./data/data_description.txt
./data/sample_submission.csv


In [69]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

train_df.shape, test_df.shape

((1460, 81), (1459, 80))

In [70]:
y = np.log1p(train_df["SalePrice"])
X = train_df.drop(columns=["SalePrice"])

X_train_raw, X_valid_raw, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_raw.shape, X_valid_raw.shape, y_train.shape, y_valid.shape

((1168, 80), (292, 80), (1168,), (292,))

In [71]:
def fill_missing_all_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    cat_cols = df.select_dtypes(include=["object", "category", "string"]).columns
    num_cols = df.select_dtypes(include=[np.number]).columns

    df[cat_cols] = df[cat_cols].fillna("None")
    df[num_cols] = df[num_cols].fillna(0)
    return df

In [72]:
X_train_f = fill_missing_all_columns(X_train_raw)
X_valid_f = fill_missing_all_columns(X_valid_raw)
X_test_f = fill_missing_all_columns(test_df)

X_train_enc = pd.get_dummies(X_train_f, drop_first=False)
X_valid_enc = pd.get_dummies(X_valid_f, drop_first=False)
X_test_enc = pd.get_dummies(X_test_f, drop_first=False)

X_valid_enc = X_valid_enc.reindex(columns=X_train_enc.columns, fill_value=0)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

X_train_enc = X_train_enc.fillna(0)
X_valid_enc = X_valid_enc.fillna(0)
X_test_enc = X_test_enc.fillna(0)

In [73]:
print("Encoded train shape:", X_train_enc.shape)
print("Encoded valid shape:", X_valid_enc.shape)
print("Encoded test shape:", X_test_enc.shape)


nonzero_share = (X_train_enc != 0).sum().sum() / (X_train_enc.shape[0] * X_train_enc.shape[1])
print(f"Non-zero share in train matrix: {nonzero_share:.4f}")

Encoded train shape: (1168, 302)
Encoded valid shape: (292, 302)
Encoded test shape: (1459, 302)
Non-zero share in train matrix: 0.2248


In [74]:
ols = LinearRegression()
ols.fit(X_train_enc, y_train)

pred_train_ols = ols.predict(X_train_enc)
pred_valid_ols = ols.predict(X_valid_enc)

rmse_train_ols = root_mean_squared_error(y_train, pred_train_ols)
rmse_valid_ols = root_mean_squared_error(y_valid, pred_valid_ols)

print(f"OLS Train RMSE (log): {rmse_train_ols:.6f}")
print(f"OLS Hold-out RMSE (log): {rmse_valid_ols:.6f}")
print(f"OLS Generalization gap: {rmse_valid_ols - rmse_train_ols:.6f}")

OLS Train RMSE (log): 0.091689
OLS Hold-out RMSE (log): 0.132367
OLS Generalization gap: 0.040678


In [75]:
ols_valid_price = np.expm1(pred_valid_ols)
actual_valid_price = np.expm1(y_valid)

rmse_valid_ols_dollars = root_mean_squared_error(actual_valid_price, ols_valid_price)
print(f"OLS Hold-out RMSE ($): {rmse_valid_ols_dollars:,.2f}")

pd.DataFrame({
    "actual": actual_valid_price,
    "predicted_ols": ols_valid_price,
}).head(10)

OLS Hold-out RMSE ($): 22,999.54


,actual,predicted_ols
892,154500.0,153713.620484
1105,325000.0,342188.571150
413,115000.0,100372.813453
522,159000.0,165989.893179
1036,315500.0,310024.330625
614,75500.0,79331.032593
218,311500.0,249238.761162
1160,146000.0,146765.626132
649,84500.0,74949.993656
887,135500.0,142737.645531


In [76]:
ridge_pipe = Pipeline([
    ("scaler", "passthrough"),
    ("regressor", Ridge(random_state=42)),
])

param_grid = {
    "scaler": [StandardScaler(), MinMaxScaler(), "passthrough"],
    "regressor__alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [77]:
grid = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    refit=True,
)

grid.fit(X_train_enc, y_train)

print("Best params:", grid.best_params_)
print(f"Best CV RMSE (log): {-grid.best_score_:.6f}")

Best params: {'regressor__alpha': 10.0, 'scaler': 'passthrough'}
Best CV RMSE (log): 0.141831


In [78]:
best_ridge = grid.best_estimator_

pred_train_ridge = best_ridge.predict(X_train_enc)
pred_valid_ridge = best_ridge.predict(X_valid_enc)

rmse_train_ridge = root_mean_squared_error(y_train, pred_train_ridge)
rmse_valid_ridge = root_mean_squared_error(y_valid, pred_valid_ridge)

print(f"Ridge Train RMSE (log): {rmse_train_ridge:.6f}")
print(f"Ridge Hold-out RMSE (log): {rmse_valid_ridge:.6f}")
print(f"Ridge Generalization gap: {rmse_valid_ridge - rmse_train_ridge:.6f}")

Ridge Train RMSE (log): 0.110160
Ridge Hold-out RMSE (log): 0.135571
Ridge Generalization gap: 0.025411


In [79]:
comparison = pd.DataFrame({
    "Model": ["OLS", "Ridge (CV tuned)"],
    "Train_RMSE_log": [rmse_train_ols, rmse_train_ridge],
    "Holdout_RMSE_log": [rmse_valid_ols, rmse_valid_ridge],
    "Generalization_Gap": [
        rmse_valid_ols - rmse_train_ols,
        rmse_valid_ridge - rmse_train_ridge,
    ],
    "CV_RMSE_log": [np.nan, -grid.best_score_],
})

comparison.sort_values("Holdout_RMSE_log")

,Model,Train_RMSE_log,Holdout_RMSE_log,Generalization_Gap,CV_RMSE_log
0,OLS,0.091689,0.132367,0.040678,NaN
1,Ridge (CV tuned),0.110160,0.135571,0.025411,0.141831


In [80]:

X_full = train_df.drop(columns=["SalePrice"])
y_full = np.log1p(train_df["SalePrice"])

X_full_f = fill_missing_all_columns(X_full)
X_test_final_f = fill_missing_all_columns(test_df)

X_full_enc = pd.get_dummies(X_full_f, drop_first=False)
X_test_final_enc = pd.get_dummies(X_test_final_f, drop_first=False)
X_test_final_enc = X_test_final_enc.reindex(columns=X_full_enc.columns, fill_value=0)

X_full_enc = X_full_enc.fillna(0)
X_test_final_enc = X_test_final_enc.fillna(0)

final_model = grid.best_estimator_
final_model.fit(X_full_enc, y_full)

test_pred_log = final_model.predict(X_test_final_enc)
test_pred_price = np.expm1(test_pred_log)

print("Computed test predictions with best Ridge model.")
pd.DataFrame({"Id": test_df["Id"], "PredictedSalePrice": test_pred_price}).head()

Computed test predictions with best Ridge model.


,Id,PredictedSalePrice
0,1461,112603.509327
1,1462,145234.578601
2,1463,168561.605567
3,1464,192513.640568
4,1465,193716.846524


In [81]:
print("Best Ridge config:", grid.best_params_)
print("\nComparison table:")
comparison

Best Ridge config: {'regressor__alpha': 10.0, 'scaler': 'passthrough'}

Comparison table:


,Model,Train_RMSE_log,Holdout_RMSE_log,Generalization_Gap,CV_RMSE_log
0,OLS,0.091689,0.132367,0.040678,NaN
1,Ridge (CV tuned),0.110160,0.135571,0.025411,0.141831


In [82]:

ols_fold_rmse = []
ridge_fold_rmse = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train_enc), start=1):
    X_tr = X_train_enc.iloc[tr_idx]
    X_va = X_train_enc.iloc[va_idx]
    y_tr = y_train.iloc[tr_idx]
    y_va = y_train.iloc[va_idx]

    # OLS
    ols_cv_model = LinearRegression()
    ols_cv_model.fit(X_tr, y_tr)
    ols_pred = ols_cv_model.predict(X_va)
    ols_rmse = root_mean_squared_error(y_va, ols_pred)
    ols_fold_rmse.append(ols_rmse)

    # Ridge with best hyperparameters from grid
    ridge_alpha = grid.best_params_["regressor__alpha"]
    ridge_scaler = grid.best_params_["scaler"]
    ridge_cv_model = Pipeline([
        ("scaler", ridge_scaler),
        ("regressor", Ridge(alpha=ridge_alpha, random_state=42)),
    ])
    ridge_cv_model.fit(X_tr, y_tr)
    ridge_pred = ridge_cv_model.predict(X_va)
    ridge_rmse = root_mean_squared_error(y_va, ridge_pred)
    ridge_fold_rmse.append(ridge_rmse)

    print(
        f"Fold {fold}: OLS RMSE={ols_rmse:.6f} | "
        f"Ridge RMSE={ridge_rmse:.6f}"
    )

kfold_summary = pd.DataFrame({
    "Model": ["OLS (5-fold)", "Ridge (5-fold, tuned alpha)"],
    "Mean_RMSE_log": [np.mean(ols_fold_rmse), np.mean(ridge_fold_rmse)],
    "Std_RMSE_log": [np.std(ols_fold_rmse), np.std(ridge_fold_rmse)],
})

print("\nKFold summary:")
display(kfold_summary.sort_values("Mean_RMSE_log"))

Fold 1: OLS RMSE=0.145338 | Ridge RMSE=0.135554
Fold 2: OLS RMSE=0.162106 | Ridge RMSE=0.139584
Fold 3: OLS RMSE=0.210766 | Ridge RMSE=0.197619
Fold 4: OLS RMSE=0.145269 | Ridge RMSE=0.117221
Fold 5: OLS RMSE=0.162410 | Ridge RMSE=0.119178

KFold summary:


,Model,Mean_RMSE_log,Std_RMSE_log
1,"Ridge (5-fold, tuned alpha)",0.141831,0.029242
0,OLS (5-fold),0.165178,0.024022


In [13]:
import os

os.environ["MLFLOW_TRACKING_USERNAME"] = "ntsuk22"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "6992bcec71bb4d142dc9fa25678fc8c81ab527a4"

In [14]:
import mlflow

mlflow.set_tracking_uri("https://dagshub.com/ntsuk22/House_Prices.mlflow")
mlflow.set_experiment("Final_Model")

2026/04/13 17:24:15 INFO mlflow.tracking.fluent: Experiment with name 'Final_Model' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/91f89d295e4a485087b1de9f49ddf176', creation_time=1776086657519, experiment_id='3', last_update_time=1776086657519, lifecycle_stage='active', name='Final_Model', tags={}, trace_location=None, workspace='default'>

In [ ]:
with mlflow.start_run():

   
    mlflow.log_param("regressor__alpha", 10.0)
    mlflow.log_param("scaler", "passthrough")


    mlflow.log_metric("best_cv_rmse_log", 0.141831)
    mlflow.log_metric("rmse_holdout_dollars", 22999.54)
    mlflow.log_metric("rmse_log_holdout", 0.135571)
    mlflow.log_metric("rmse_log_train", 0.11016)

🏃 View run powerful-shrew-371 at: https://dagshub.com/ntsuk22/House_Prices.mlflow/#/experiments/3/runs/1186134cd8764888bc759cd9d115d97b
🧪 View experiment at: https://dagshub.com/ntsuk22/House_Prices.mlflow/#/experiments/3


In [19]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import Ridge

import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature


RUN_ID = "1186134cd8764888bc759cd9d115d97b"
TRACKING_URI = "https://dagshub.com/ntsuk22/House_Prices.mlflow"
REGISTERED_MODEL_NAME = "HousePricesRidgeModel"
ARTIFACT_PATH = "model"


train_df = pd.read_csv("./data/train.csv")

def fill_missing_all_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    cat_cols = df.select_dtypes(include=["object", "category", "string"]).columns
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[cat_cols] = df[cat_cols].fillna("None")
    df[num_cols] = df[num_cols].fillna(0)
    return df

y = np.log1p(train_df["SalePrice"])
X = train_df.drop(columns=["SalePrice"])

X_train_raw, X_valid_raw, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_f = fill_missing_all_columns(X_train_raw)
X_valid_f = fill_missing_all_columns(X_valid_raw)

X_train_enc = pd.get_dummies(X_train_f, drop_first=False)
X_valid_enc = pd.get_dummies(X_valid_f, drop_first=False)

X_valid_enc = X_valid_enc.reindex(columns=X_train_enc.columns, fill_value=0)

X_train_enc = X_train_enc.fillna(0)
X_valid_enc = X_valid_enc.fillna(0)


X_full = train_df.drop(columns=["SalePrice"])
y_full = np.log1p(train_df["SalePrice"])
X_full_f = fill_missing_all_columns(X_full)
X_full_enc = pd.get_dummies(X_full_f, drop_first=False).fillna(0)


ridge_pipe = Pipeline([
    ("scaler", "passthrough"),
    ("regressor", Ridge(random_state=42)),
])

param_grid = {
    "scaler": [StandardScaler(), MinMaxScaler(), "passthrough"],
    "regressor__alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    refit=True,
)

grid.fit(X_train_enc, y_train)

final_model = grid.best_estimator_
final_model.fit(X_full_enc, y_full)

print("Best params:", grid.best_params_)


mlflow.set_tracking_uri(TRACKING_URI)

input_example = X_full_enc.head(5)
pred_example = final_model.predict(input_example)
signature = infer_signature(input_example, pred_example)

with mlflow.start_run(run_id=RUN_ID):
    mlflow.sklearn.log_model(
        sk_model=final_model,
        artifact_path=ARTIFACT_PATH,
        signature=signature,
        input_example=input_example,
        registered_model_name=REGISTERED_MODEL_NAME,
    )

print(f"Registered model: {REGISTERED_MODEL_NAME}")
print(f"Model URI: runs:/{RUN_ID}/{ARTIFACT_PATH}")

Best params: {'regressor__alpha': 10.0, 'scaler': 'passthrough'}


/home/MyDocuments (ubuntu)/ML/Assignmentas/House_Prices/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/13 17:41:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 17:41:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cl

🏃 View run FINAL_MODEL regression2 at: https://dagshub.com/ntsuk22/House_Prices.mlflow/#/experiments/3/runs/1186134cd8764888bc759cd9d115d97b
🧪 View experiment at: https://dagshub.com/ntsuk22/House_Prices.mlflow/#/experiments/3
Registered model: HousePricesRidgeModel
Model URI: runs:/1186134cd8764888bc759cd9d115d97b/model
